In [28]:

import pandas as pd 
import numpy as np 

# Load churnguard_data.csv
df = pd.read_csv("churnguard_data.csv")

# Drop the customerID column
df = df.drop(["customerID"] , axis = 1)

# Remove duplicate rows
df = df.drop_duplicates()
# print(df.duplicated().sum())

# Strip whitespace from gender and PaymentMethod using .str.strip()
cols_del_whitespace = ["gender" , "PaymentMethod"]
df[cols_del_whitespace] = df[cols_del_whitespace].apply(lambda x: x.str.strip())
# df["PhoneService"].unique()

# Standardise casing — convert Churn, PhoneService, and PaperlessBilling to title case using .str.strip().str.title()
title_case = ['Churn' , 'PhoneService' , "PaperlessBilling"]
df[title_case] = df[title_case].apply(lambda x: x.str.strip().str.title())


# Fix Contract — map all variations to one of three valid values:
# Month-to-month, One year, Two year
contract_map = {
    "Month-to-month": "Month-to-month",
    "One year":       "One year",      
    "Two year":       "Two year",      
    "month-to-month": "Month-to-month",
    "month to month": "Month-to-month",
    "Monthly":        "Month-to-month",
    "One Year":       "One year",
    "one year":       "One year",
    "1 year":         "One year",
    "Two Year":       "Two year",
    "two year":       "Two year",
    "2 year":         "Two year",
    "2 Year":         "Two year",
}

df["Contract"] = df["Contract"].str.strip().map(contract_map)


# Fix InternetService — map all variations to one of three valid values:
# DSL, Fiber optic, No
internet_map = { "dsl":"DSL" , 
                 "fiberoptic":"Fiber optic" ,
                 "fiber optic":"Fiber optic",
                 "no":"No",
                 "fibre optic":"Fiber optic"}

df['InternetService'] = df['InternetService'].str.strip().str.lower().map(internet_map)
df['InternetService'] = df['InternetService'].fillna(df['InternetService'].mode()[0])

# Fix TotalCharges — convert to numeric using pd.to_numeric(..., errors='coerce') so junk becomes NaN
# Remove rows where tenure is zero or negative
# Remove rows where MonthlyCharges is less than 10 or greater than 200
df["TotalCharges"] = pd.to_numeric( df["TotalCharges"], errors="coerce")
df.drop(df[df['tenure'] <= 0].index, inplace=True)
df.drop(df[(df['MonthlyCharges'] < 10) | (df['MonthlyCharges'] >200)].index, inplace=True)

# Fill missing values:
# MonthlyCharges → column mean
# TotalCharges → column mean
df['MonthlyCharges'] = df['MonthlyCharges'].fillna(df['MonthlyCharges'].mean())
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].mean())

# tenure → column median (use integer rounding)
df['tenure'] = df['tenure'].fillna(df['tenure'].median()).astype(int)

# Print the shape of the cleaned DataFrame
print(f"The shape of the Data frame after cleaning :- {df.shape}")
print("=" * 55)

# Print missing value counts to confirm all issues are resolved
print("After completion of cleaning the data , the no.of missing values are:- ")
print(f'{df.isnull().sum()}')

The shape of the Data frame after cleaning :- (980, 11)
After completion of cleaning the data , the no.of missing values are:- 
gender              0
SeniorCitizen       0
tenure              0
PhoneService        0
InternetService     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64


In [29]:
df

,gender,SeniorCitizen,tenure,PhoneService,InternetService,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Male,0,21,Yes,Fiber optic,Month-to-month,No,Credit card,29.73,600.01,Yes
1,Male,0,55,Yes,Fiber optic,Two year,Yes,Bank transfer,46.32,2515.48,No
2,Female,1,46,Yes,Fiber optic,Month-to-month,No,Mailed check,87.06,4153.97,Yes
3,Female,1,63,Yes,Fiber optic,Month-to-month,Yes,Mailed check,56.97,3641.30,Yes
4,Female,0,8,Yes,DSL,Month-to-month,No,Electronic check,39.69,309.79,Yes
...,...,...,...,...,...,...,...,...,...,...,...
1025,Female,0,26,Yes,DSL,One year,Yes,Electronic check,51.68,1369.73,No
1026,Male,0,49,Yes,Fiber optic,Two year,Yes,Electronic check,85.39,4305.40,No
1027,Male,1,69,Yes,DSL,Two year,Yes,Electronic check,95.00,6689.62,No
1028,Female,1,3,Yes,No,Two year,Yes,Electronic check,25.08,77.20,No


In [30]:
df.columns

Index(['gender', 'SeniorCitizen', 'tenure', 'PhoneService', 'InternetService',
       'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges',
       'TotalCharges', 'Churn'],
      dtype='str')

In [31]:
df.info()

<class 'pandas.DataFrame'>
Index: 980 entries, 0 to 1029
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            980 non-null    str    
 1   SeniorCitizen     980 non-null    int64  
 2   tenure            980 non-null    int64  
 3   PhoneService      980 non-null    str    
 4   InternetService   980 non-null    str    
 5   Contract          980 non-null    str    
 6   PaperlessBilling  980 non-null    str    
 7   PaymentMethod     980 non-null    str    
 8   MonthlyCharges    980 non-null    float64
 9   TotalCharges      980 non-null    float64
 10  Churn             980 non-null    str    
dtypes: float64(2), int64(2), str(7)
memory usage: 91.9 KB


# TASK - 3_Train the Model

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder



# Encode the target column Churn:
# Yes → 1, No → 0
label_enc = LabelEncoder()
df["Churn"] = label_enc.fit_transform(df['Churn'])

# Encode categorical columns using pd.get_dummies() with drop_first=True:
# gender, PhoneService, InternetService, Contract, PaperlessBilling, PaymentMethod
cat_cols = ['gender','PhoneService', 'InternetService',
       'Contract', 'PaperlessBilling', 'PaymentMethod']
df = pd.get_dummies(df , columns=cat_cols , drop_first=True)


KeyError: "None of [Index(['gender', 'PhoneService', 'InternetService', 'Contract',\n       'PaperlessBilling', 'PaymentMethod'],\n      dtype='str')] are in the [columns]"

In [ ]:

# Separate the data into:
# X — all columns except Churn
# y — the Churn column
x = df.drop(['Churn'] , axis= 1)
y = df['Churn']

# Split into train and test sets — 80% train, 20% test, random_state=42
X_train , X_test , y_train , y_test = train_test_split(x , y , test_size=0.2 , random_state=42)

# Train a LogisticRegression model with max_iter=1000
model = LogisticRegression(max_iter=1000)
model.fit(X_train , y_train)

# Print the accuracy score on the test set
# Print the classification report using classification_report with target_names=['Stay', 'Churn']